In [3]:
import numpy as np
import pandas as pd
import time
import itertools
import os, sys
# Build an absolute path from this notebook's parent directory
module_path = os.path.abspath('src')

# Add to sys.path if not already present
if module_path not in sys.path:
    sys.path.append(module_path)

# from gridsearch import run_all, final_predict,pipeline_knn,pipeline_blend,make_folds, rmse
# from ameliorations import UserContentRidge
from compareKnn import _sf_item_rows, SimilarityFusion, sf_prepare, _sur_neighbors, _sf_combine,_sf_parts, _sf_item_sim, _sir_neighbors,parse_movies,_sf_xcol,_ratio
from KnnBoosted import KnnBoosted
from knnFinal import FusionKNN

np.set_printoptions(legacy='1.25')

In [3]:
trainData = np.load("datasets/trainData.npy")
valData = np.load("datasets/valData.npy")
movieData = np.load("datasets/namesngenre.npy")
testData = np.load("datasets/testData.npy")

R_cv = np.where(np.isnan(trainData), valData, trainData)

In [3]:
SF_KNN_PARAMS = ["sim", "signed", "alpha", "w_genre", "tau", "select",
                 "k_user", "k_movie", "k_user_cross", "k_movie_cross", "budget",
                 "comb", "lam", "delta"]
SF_PARAMS = ["reg_u", "reg_i"] + SF_KNN_PARAMS

class SFGridSearch:
    """Évalue toute une grille sur les notes de validation d'un découpage train / val."""

    def __init__(self, trainData, valData, feats, reg_u=15.0, reg_i=10.0, n_iter=3,
                 sim="paper", RMIN=0.5, RMAX=5.0):
        self.sim, self.RMIN, self.RMAX = sim, RMIN, RMAX
        self.prep = sf_prepare(trainData, feats, reg_u, reg_i, n_iter, sim)
        self.u, self.i = np.where(~np.isnan(valData))
        self.y = valData[self.u, self.i]
        p = self.prep
        self.base = p["mu"] + p["b_u"][self.u] + p["b_i"][self.i]
        self.rows, self.r = np.unique(self.i, return_inverse=True)
        self.item_rows = _sf_item_rows(p, self.rows)            # indépendant de alpha / w_genre / tau

    def run(self, N=None, k_user=None, k_movie=None, N_cross=None,
            k_user_cross=None, k_movie_cross=None, select=None,
            lam=(0.7,), delta=(0.7,), comb=("eucl",), signed=False,
            alpha=(1.0,), w_genre=(0.7,), tau=(10.0,), raw=False, verbose=True):
        p = self.prep
        U1, I1 = p["ordU"].shape[1], p["M"].shape[1] - 1
        paper = N is not None
        sel = select or ("rated" if paper else "global")

        if paper:
            ku = ki = sorted({int(min(n, U1, I1)) for n in N})
            xu = xi = sorted({int(min(n, ku[-1])) for n in (N_cross or N)})
            second = xu                                   # budget de notes = mêmes valeurs
        else:
            ku = sorted({int(min(n, U1)) for n in k_user})
            ki = sorted({int(min(n, I1)) for n in k_movie})
            xu = sorted({int(min(n, ku[-1])) for n in (k_user_cross or k_user)})
            xi = sorted({int(min(n, ki[-1])) for n in (k_movie_cross or k_movie)})
            second = xi

        lams, deltas = np.asarray(lam, float), np.asarray(delta, float)
        cross = bool((deltas > 0).any())
        combs = list(comb) if cross else [None]

        t0 = time.time()
        idxU, wU = _sur_neighbors(p, self.u, self.i, ku[-1], sel, signed)
        if verbose:
            print(f"  [{self.sim}/{sel}] voisins SUR : {time.time() - t0:.1f}s")

        item_cfgs = [(a, wg, t) for a in alpha
                     for wg, t in (itertools.product(w_genre, tau) if a < 1 else [(None, None)])]
        rec = []
        for a, wg, t in item_cfgs:
            t0 = time.time()
            S = _sf_item_sim(self.item_rows, self.rows, a, wg, t)
            idxI, wI = _sir_neighbors(p["M"], S, self.r, self.u, ki[-1], sel)
            del S
            parts = _sf_parts(p, self.u, self.i, idxU, wU, idxI, wI, ku, ki, xu, xi,
                              budget=second if paper else None, cross=cross)

            for (iu, nu), (ii_, ni) in itertools.product(enumerate(ku), enumerate(ki)):
                for ja, nxu in enumerate(xu):
                    for jb, nxi in enumerate(second if paper else xi):
                        jx = _sf_xcol(ja, jb, len(second))
                        for cb in combs:
                            e = _sf_combine(parts, iu, ii_, jx, lams, deltas, cb)
                            pred = np.clip(self.base + e, self.RMIN, self.RMAX)
                            err = np.sqrt(((pred - self.y) ** 2).mean(-1))
                            for (il, l), (idl, d) in itertools.product(enumerate(lams),
                                                                       enumerate(deltas)):
                                on = d > 0
                                rec.append((self.sim, signed, a, wg, t, sel, nu, ni,
                                            nxu if on else None,
                                            (None if paper else nxi) if on else None,
                                            (nxi if paper else None) if on else None,
                                            cb if on else None, l, d, err[il, idl]))
            if verbose:
                print(f"  α={a}, w_genre={wg}, τ={t} : {time.time() - t0:.1f}s")

        df = pd.DataFrame(rec, columns=SF_KNN_PARAMS + ["RMSE"])
        return df if raw else df.drop_duplicates(subset=SF_KNN_PARAMS).sort_values("RMSE").reset_index(drop=True)

def cv_sf_grid_search(R, movieData, n_folds=5, max_fold= 5, seed=42, reg_u=(15.0,), reg_i=(10.0,),
                      sim=("paper",), n_iter=3, RMIN=0.5, RMAX=5.0, verbose=True, **grid):
    """Mêmes folds que cv_grid_search, cv_mf et oof_predictions (même seed)."""
    feats = parse_movies(pd.DataFrame(movieData))
    u, i = np.where(~np.isnan(R))
    folds = np.array_split(np.random.default_rng(seed).permutation(len(u)), n_folds)[:max_fold]

    params, scores = None, []
    for f, idx in enumerate(folds):
        t0 = time.time()
        train, val = R.copy(), np.full_like(R, np.nan)
        train[u[idx], i[idx]] = np.nan
        val[u[idx], i[idx]] = R[u[idx], i[idx]]

        dfs = []
        for ru, ri, sm in itertools.product(reg_u, reg_i, sim):
            gs = SFGridSearch(train, val, feats, ru, ri, n_iter, sm, RMIN, RMAX)
            dfs.append(gs.run(**grid, raw=True, verbose=verbose).assign(reg_u=ru, reg_i=ri))
            del gs
        df = pd.concat(dfs, ignore_index=True)
        del train, val

        if params is None:
            params = df[SF_PARAMS]
        else:
            assert params.equals(df[SF_PARAMS]), "grilles différentes entre folds"
        scores.append(df["RMSE"].to_numpy())
        print(f"fold {f + 1}/{max_fold} : {time.time() - t0:.0f}s, meilleure RMSE {df['RMSE'].min():.4f}")

    scores = np.column_stack(scores)
    res = params.assign(RMSE=scores.mean(1), std=scores.std(1),
                        **{f"fold{f}": scores[:, f] for f in range(max_fold)})
    return res.drop_duplicates(subset=SF_PARAMS).sort_values("RMSE").reset_index(drop=True)

def sf_to_params(row):
    p = row[SF_PARAMS].dropna().to_dict()
    sel = p.pop("select")
    budget = p.pop("budget", None)
    if sel == "rated" and budget is not None:            # logique papier : N unique
        p["N"], p["N_cross"] = int(p.pop("k_user")), int(budget)
        p.pop("k_movie", None); p.pop("k_user_cross", None); p.pop("k_movie_cross", None)
    else:
        for k in ("k_user", "k_movie", "k_user_cross", "k_movie_cross"):
            if k in p:
                p[k] = int(p[k])
    p["select"] = sel
    return p

In [4]:
def showParams(df: pd.DataFrame):
    exclus = ["RMSE", 'std', 'fold0', 'fold1', 'fold2', 'fold3', 'fold4', 'fold5', 'temps_s', 'Unnamed:']
    for p in df.columns:
        if p not in exclus and df[p].nunique()>1:
            res = df.groupby(p, dropna=False)["RMSE"].min().to_frame().T
            display(
                res.style.highlight_min(axis=None, props="background-color: green; font-weight: bold;")
            )

In [4]:
def observed(R):
    """Masque booléen des notes connues."""
    return ~np.isnan(R) & (R != 0)

def rmse(P, R, mask):
    return np.sqrt(np.mean((P[mask] - R[mask]) ** 2))

In [ ]:
base = dict(reg_u=[0], reg_i=[0], n_iter=1, sim=["paper"], alpha=[1.0],
            lam=np.linspace(0, 1, 11), delta=[0, 0.2, 0.4, 0.6, 0.8], comb=["eucl"])

étape0 = dict(base, N=[30, 50, 75, 100, 150])                        # papier
étape1 = dict(étape0, alpha=[1.0, 0.9, 0.8])                         # + features films
étape2 = dict(base, k_user=[20, 50, 150, 350], k_movie=[50, 300, 1500], alpha=[0.9])
étape3 = dict(étape2, k_user_cross=[10, 30], k_movie_cross=[20, 50]) # + rectangle séparé
étape4 = dict(étape3, sim=["residual"])                              # + résidus
étape5 = dict(étape4, comb=["prod"])                                 # + produit
étape6 = dict(étape5, reg_u=[15], reg_i=[10], n_iter=3)              # + régularisation
étape7 = dict(étape6, signed=True)                                   # = KnnBoosted

In [ ]:
paper_sf = cv_sf_grid_search(
    R_cv, movieData, n_folds=3, seed=42,
    reg_u=[0], reg_i=[0], n_iter=1,
    sim=["paper"],
    N=[50, 75, 100, 125, 150],
    lam=np.linspace(0, 1, 11),
    delta=[0, 0.2, 0.4],
    comb=["eucl"],
    alpha=[1.0], w_genre=[0.7], tau=[10],
)
paper_sf[SF_PARAMS + ["RMSE", "std"]].head(1)

  [paper/notes] voisins SUR : 0.7s
  α=1.0, w_genre=None, τ=None : 37.3s
fold 1/3 : 41s, meilleure RMSE 0.8930
  [paper/notes] voisins SUR : 0.6s
  α=1.0, w_genre=None, τ=None : 33.6s
fold 2/3 : 36s, meilleure RMSE 0.8994
  [paper/notes] voisins SUR : 0.8s
  α=1.0, w_genre=None, τ=None : 35.8s
fold 3/3 : 40s, meilleure RMSE 0.8988


,reg_u,reg_i,sim,alpha,w_genre,tau,mode,N,k_user,k_movie,comb,lam,delta,RMSE,std
0,0,0,paper,1.0,None,None,notes,150,None,None,None,0.7,0.0,0.897076,0.002867
1,0,0,paper,1.0,None,None,notes,125,None,None,None,0.7,0.0,0.897100,0.002862
2,0,0,paper,1.0,None,None,notes,100,None,None,None,0.7,0.0,0.897116,0.002852
3,0,0,paper,1.0,None,None,notes,75,None,None,None,0.7,0.0,0.897157,0.002839
4,0,0,paper,1.0,None,None,notes,50,None,None,None,0.7,0.0,0.897228,0.002849
5,0,0,paper,1.0,None,None,notes,150,None,None,None,0.8,0.0,0.897452,0.002793
6,0,0,paper,1.0,None,None,notes,150,None,None,None,0.6,0.0,0.897459,0.002938
7,0,0,paper,1.0,None,None,notes,125,None,None,None,0.8,0.0,0.897466,0.002790
8,0,0,paper,1.0,None,None,notes,100,None,None,None,0.8,0.0,0.897472,0.002780
9,0,0,paper,1.0,None,None,notes,75,None,None,None,0.8,0.0,0.897488,0.002773


In [ ]:
paper_sf.to_csv("gridResult/paper_sf1.csv")
paper_sf[SF_PARAMS + ["RMSE", "std"]].head(10)
showParams(paper_sf)


NameError: name 'paper_sf' is not defined

In [33]:
mvF_sf = cv_sf_grid_search(
    R_cv, movieData, n_folds=6, seed=42,
    reg_u=[0], reg_i=[0], n_iter=1,
    sim=["paper"],
    N=[25,40,50],
    lam=[0.45,0.5,0.6],
    delta=[0, 0.1],
    comb=["eucl"],
    alpha=[0.1, 0.2,0.3], w_genre=[0.3], tau=[5],
)
mvF_sf.to_csv("gridResult/mvF_sf3.csv")

mvF_sf[SF_PARAMS + ["RMSE", "std"]].head(1)

  [paper/notes] voisins SUR : 0.4s
  α=0.1, w_genre=0.3, τ=5 : 2.4s
  α=0.2, w_genre=0.3, τ=5 : 2.2s
  α=0.3, w_genre=0.3, τ=5 : 2.3s
fold 1/6 : 9s, meilleure RMSE 0.8691
  [paper/notes] voisins SUR : 0.3s
  α=0.1, w_genre=0.3, τ=5 : 2.2s
  α=0.2, w_genre=0.3, τ=5 : 2.2s
  α=0.3, w_genre=0.3, τ=5 : 2.2s
fold 2/6 : 8s, meilleure RMSE 0.8861
  [paper/notes] voisins SUR : 0.3s
  α=0.1, w_genre=0.3, τ=5 : 2.2s
  α=0.2, w_genre=0.3, τ=5 : 2.2s
  α=0.3, w_genre=0.3, τ=5 : 2.2s
fold 3/6 : 8s, meilleure RMSE 0.8833
  [paper/notes] voisins SUR : 0.3s
  α=0.1, w_genre=0.3, τ=5 : 2.2s
  α=0.2, w_genre=0.3, τ=5 : 2.2s
  α=0.3, w_genre=0.3, τ=5 : 2.2s
fold 4/6 : 8s, meilleure RMSE 0.8834
  [paper/notes] voisins SUR : 0.3s
  α=0.1, w_genre=0.3, τ=5 : 2.5s
  α=0.2, w_genre=0.3, τ=5 : 2.2s
  α=0.3, w_genre=0.3, τ=5 : 2.2s
fold 5/6 : 9s, meilleure RMSE 0.8829
  [paper/notes] voisins SUR : 0.3s
  α=0.1, w_genre=0.3, τ=5 : 2.3s
  α=0.2, w_genre=0.3, τ=5 : 2.2s
  α=0.3, w_genre=0.3, τ=5 : 2.2s
fold 6/6 : 

,reg_u,reg_i,sim,alpha,w_genre,tau,mode,N,k_user,k_movie,comb,lam,delta,RMSE,std
0,0,0,paper,0.3,0.3,5,notes,25,None,None,None,0.6,0.0,0.88172,0.005723


In [ ]:

showParams(mvF_sf)
mvF_sf[SF_PARAMS + ["RMSE", "std"]].head(1)


NameError: name 'mvF_sf' is not defined

In [14]:
kk = cv_sf_grid_search(
    R_cv, movieData, n_folds=6,max_fold=2, seed=42,
    reg_u=[0], reg_i=[0], n_iter=1,
    sim=["paper"],
    k_movie=[400,800, 1300],
    k_user=[100,200,300,400],
    lam=[0.2, 0.4, 0.6],
    delta=[0, 0.1, 0.3],
    comb=["eucl"],
    alpha=[0.1], w_genre=[0.3], tau=[5],
)
kk.to_csv("gridResult/kk1.csv")

kk[SF_PARAMS + ["RMSE", "std"]].head(1)

  [paper/global] voisins SUR : 0.1s
  α=0.1, w_genre=0.3, τ=5 : 530.5s
fold 1/2 : 535s, meilleure RMSE 0.8720
  [paper/global] voisins SUR : 0.0s
  α=0.1, w_genre=0.3, τ=5 : 561.2s
fold 2/2 : 563s, meilleure RMSE 0.8887


,reg_u,reg_i,sim,signed,alpha,w_genre,tau,select,k_user,k_movie,k_user_cross,k_movie_cross,budget,comb,lam,delta,RMSE,std
0,0,0,paper,False,0.1,0.3,5,global,400,800,NaN,NaN,None,None,0.6,0.0,0.88042,0.008306


In [ ]:
showParams(kk)
kk[SF_PARAMS + ["RMSE", "std"]].head(1)

k_user,100,200,300,400
RMSE,0.884417,0.882278,0.880878,0.880420


k_movie,400,800,1300
RMSE,0.882832,0.880420,0.880535


k_user_cross,100.000000,200.000000,300.000000,400.000000,nan
RMSE,0.882759,0.882933,0.882940,0.882979,0.880420


k_movie_cross,400.000000,800.000000,1300.000000,nan
RMSE,0.883626,0.883122,0.882759,0.880420


lam,0.200000,0.400000,0.600000
RMSE,0.886334,0.882227,0.880420


delta,0.000000,0.100000,0.300000
RMSE,0.880420,0.882759,0.888881


In [16]:
kk_cross = cv_sf_grid_search(
    R_cv, movieData, n_folds=6,max_fold=2, seed=42,
    reg_u=[0], reg_i=[0], n_iter=1,
    sim=["paper"],
    k_movie=[800,900],
    k_user=[300,400],
    k_user_cross=[50,100,200],
    k_movie_cross = [100,200,400],
    lam=[0.6],
    delta=[0.1],
    comb=["eucl"],
    alpha=[0.1], w_genre=[0.3], tau=[5],
)
kk_cross.to_csv("gridResult/kk_cross1.csv")

kk_cross[SF_PARAMS + ["RMSE", "std"]].head(1)

  [paper/global] voisins SUR : 0.1s
  α=0.1, w_genre=0.3, τ=5 : 63.3s
fold 1/2 : 66s, meilleure RMSE 0.8749
  [paper/global] voisins SUR : 0.0s
  α=0.1, w_genre=0.3, τ=5 : 95.4s
fold 2/2 : 97s, meilleure RMSE 0.8917


,reg_u,reg_i,sim,signed,alpha,w_genre,tau,select,k_user,k_movie,k_user_cross,k_movie_cross,budget,comb,lam,delta,RMSE,std
0,0,0,paper,False,0.1,0.3,5,global,400,900,50,400,None,eucl,0.6,0.1,0.883298,0.008434


In [ ]:
showParams(kk_cross)

k_user,300,400
RMSE,0.883659,0.883298


k_movie,800,900
RMSE,0.883331,0.883298


k_user_cross,50,100,200
RMSE,0.883298,0.883602,0.883837


k_movie_cross,100,200,400
RMSE,0.883984,0.883830,0.883298


In [20]:
residu = cv_sf_grid_search(
    R_cv, movieData, n_folds=6,max_fold=2, seed=42,
    reg_u=[0], reg_i=[0], n_iter=1,
    sim=["residual"],
    k_movie=[600,900],
    k_user=[100,300,500],
    k_user_cross=[50,100,200],
    k_movie_cross = [100,200,400],
    lam=[0.4,0.6],
    delta=[0.1,0.2],
    comb=["eucl"],
    alpha=[0.1,0.3], w_genre=[0.3], tau=[5],
)
residu.to_csv("gridResult/residu1.csv")

residu[SF_PARAMS + ["RMSE", "std"]].head(1)

  [residual/global] voisins SUR : 0.1s
  α=0.1, w_genre=0.3, τ=5 : 64.2s
  α=0.3, w_genre=0.3, τ=5 : 68.5s
fold 1/2 : 136s, meilleure RMSE 0.8821
  [residual/global] voisins SUR : 0.0s
  α=0.1, w_genre=0.3, τ=5 : 64.0s
  α=0.3, w_genre=0.3, τ=5 : 73.4s
fold 2/2 : 138s, meilleure RMSE 0.8948


,reg_u,reg_i,sim,signed,alpha,w_genre,tau,select,k_user,k_movie,k_user_cross,k_movie_cross,budget,comb,lam,delta,RMSE,std
0,0,0,residual,False,0.3,0.3,5,global,300,900,100,400,None,eucl,0.4,0.1,0.888439,0.006344


In [ ]:
showParams(residu)

alpha,0.100000,0.300000
RMSE,0.888633,0.888439


k_user,100,300,500
RMSE,0.892230,0.888439,0.888439


k_movie,600,900
RMSE,0.889970,0.888439


k_user_cross,50,100,200
RMSE,0.888849,0.888439,0.888535


k_movie_cross,100,200,400
RMSE,0.888535,0.888691,0.888439


lam,0.400000,0.600000
RMSE,0.888439,0.891898


delta,0.100000,0.200000
RMSE,0.888439,0.889403


In [22]:
simProd = cv_sf_grid_search(
    R_cv, movieData, n_folds=6,max_fold=2, seed=42,
    reg_u=[0], reg_i=[0], n_iter=1,
    sim=["residual"],
    k_movie=[600,900],
    k_user=[100,300,500],
    k_user_cross=[50,100,200],
    k_movie_cross = [100,200,400],
    lam=[0.4,0.6],
    delta=[0.1,0.2],
    comb=["prod"],
    alpha=[0.1,0.3], w_genre=[0.3], tau=[5],
)
simProd.to_csv("gridResult/simProd1.csv")

simProd[SF_PARAMS + ["RMSE", "std"]].head(1)

  [residual/global] voisins SUR : 0.1s
  α=0.1, w_genre=0.3, τ=5 : 67.5s
  α=0.3, w_genre=0.3, τ=5 : 61.4s
fold 1/2 : 131s, meilleure RMSE 0.8821
  [residual/global] voisins SUR : 0.0s
  α=0.1, w_genre=0.3, τ=5 : 58.2s
  α=0.3, w_genre=0.3, τ=5 : 65.4s
fold 2/2 : 125s, meilleure RMSE 0.8948


,reg_u,reg_i,sim,signed,alpha,w_genre,tau,select,k_user,k_movie,k_user_cross,k_movie_cross,budget,comb,lam,delta,RMSE,std
0,0,0,residual,False,0.3,0.3,5,global,300,900,100,400,None,prod,0.4,0.1,0.888473,0.006374


In [ ]:
showParams(simProd)

alpha,0.100000,0.300000
RMSE,0.888646,0.888473


k_user,100,300,500
RMSE,0.892258,0.888473,0.888473


k_movie,600,900
RMSE,0.890010,0.888473


k_user_cross,50,100,200
RMSE,0.888872,0.888473,0.888564


k_movie_cross,100,200,400
RMSE,0.888564,0.888712,0.888473


lam,0.400000,0.600000
RMSE,0.888473,0.891940


delta,0.100000,0.200000
RMSE,0.888473,0.889488


In [24]:
regu = cv_sf_grid_search(
    R_cv, movieData, n_folds=6,max_fold=2, seed=42,
    reg_u=[5,10,15], reg_i=[5,10,15], n_iter=5,
    sim=["residual"],
    k_movie=[600,900],
    k_user=[100,300,500],
    k_user_cross=[50,100,200],
    k_movie_cross = [100,200,400],
    lam=[0.4,0.6],
    delta=[0.1,0.2],
    comb=["prod"],
    alpha=[0.1,0.3], w_genre=[0.3], tau=[5],
)
regu.to_csv("gridResult/regu1.csv")

regu[SF_PARAMS + ["RMSE", "std"]].head(1)

  [residual/global] voisins SUR : 0.0s
  α=0.1, w_genre=0.3, τ=5 : 59.9s
  α=0.3, w_genre=0.3, τ=5 : 72.9s
  [residual/global] voisins SUR : 0.1s
  α=0.1, w_genre=0.3, τ=5 : 78.5s
  α=0.3, w_genre=0.3, τ=5 : 65.3s
  [residual/global] voisins SUR : 0.1s
  α=0.1, w_genre=0.3, τ=5 : 56.8s
  α=0.3, w_genre=0.3, τ=5 : 57.7s
  [residual/global] voisins SUR : 0.0s
  α=0.1, w_genre=0.3, τ=5 : 58.2s
  α=0.3, w_genre=0.3, τ=5 : 57.6s
  [residual/global] voisins SUR : 0.0s
  α=0.1, w_genre=0.3, τ=5 : 61.6s
  α=0.3, w_genre=0.3, τ=5 : 65.2s
  [residual/global] voisins SUR : 0.0s
  α=0.1, w_genre=0.3, τ=5 : 63.1s
  α=0.3, w_genre=0.3, τ=5 : 64.8s
  [residual/global] voisins SUR : 0.1s
  α=0.1, w_genre=0.3, τ=5 : 68.9s
  α=0.3, w_genre=0.3, τ=5 : 64.8s
  [residual/global] voisins SUR : 0.1s
  α=0.1, w_genre=0.3, τ=5 : 64.7s
  α=0.3, w_genre=0.3, τ=5 : 67.2s
  [residual/global] voisins SUR : 0.1s
  α=0.1, w_genre=0.3, τ=5 : 63.8s
  α=0.3, w_genre=0.3, τ=5 : 62.4s
fold 1/2 : 1167s, meilleure RMSE 0.85

,reg_u,reg_i,sim,signed,alpha,w_genre,tau,select,k_user,k_movie,k_user_cross,k_movie_cross,budget,comb,lam,delta,RMSE,std
0,10,10,residual,False,0.3,0.3,5,global,500,900,100,100,None,prod,0.4,0.1,0.856381,0.005865


In [ ]:
showParams(regu)

reg_u,5,10,15
RMSE,0.856502,0.856381,0.856470


reg_i,5,10,15
RMSE,0.856618,0.856381,0.856879


alpha,0.100000,0.300000
RMSE,0.857005,0.856381


k_user,100,300,500
RMSE,0.859325,0.856404,0.856381


k_movie,600,900
RMSE,0.858027,0.856381


k_user_cross,50,100,200
RMSE,0.856597,0.856381,0.856512


k_movie_cross,100,200,400
RMSE,0.856381,0.856558,0.856418


lam,0.400000,0.600000
RMSE,0.856381,0.860185


delta,0.100000,0.200000
RMSE,0.856381,0.857308


In [10]:
paper_params = {"N":150, "lam":0.7, "delta":0.1} #* RMSE : 0.860
movieF_params = {"N":25, "lam":0.7, "delta":0.1, "alpha":0.3} #* RMSE : 0.859
kk_params = {"k_user":400, "k_movie":800, "delta":0.1, "alpha":0.3, "lam":0.7} #* RMSE : 0.856
kk_cross_params = {"k_user":400, "k_movie":900, "delta":0.1, "alpha":0.3, "lam":0.7, "k_user_cross":50, "k_movie_cross":400} #* RMSE : 0.855
residu_params = {"k_user":300, "k_movie":900, "delta":0.1, "alpha":0.3, "lam":0.4, "k_user_cross":100, "k_movie_cross":400, "sim":"residual"} #* RMSE : 0.867
simProd_params = {"k_user":300, "k_movie":900, "delta":0.1, "alpha":0.3, "lam":0.4, "k_user_cross":100, "k_movie_cross":400, "sim":"residual", "comb":"prod"} #* RMSE : 0.867
simProd_params2 = {"k_user":300, "k_movie":900, "delta":0.1, "alpha":0.3, "lam":0.4, "k_user_cross":100, "k_movie_cross":400, "sim":"paper", "comb":"prod"} #* RMSE : 0.867
regu_params = {"k_user":500, "k_movie":900, "delta":0.1, "alpha":0.3, "lam":0.4, "k_user_cross":100, "k_movie_cross":400, "sim":"residual", "comb":"prod", "reg_u":10, "reg_i":10,"n_iter":5 } #* RMSE : 0.837
regu_params2 = {"k_user":500, "k_movie":900, "delta":0.1, "alpha":0.3, "lam":0.4, "k_user_cross":100, "k_movie_cross":400, "sim":"paper", "comb":"prod", "reg_u":10, "reg_i":10,"n_iter":5 } #* RMSE : 0.831
regu_params3 = {"k_user":500, "k_movie":900, "delta":0.1, "alpha":0.3, "lam":0.4, "k_user_cross":100, "k_movie_cross":400, "sim":"paper", "comb":"eucl", "reg_u":10, "reg_i":10,"n_iter":5 } #* RMSE : 0.831

listParams = [
    # paper_params,
    # movieF_params,
    # kk_params,
    kk_cross_params,
    # residu_params,
    # simProd_params,
    simProd_params2,
    # regu_params,
    # regu_params2,
]

# for params in listParams:
#     model = SimilarityFusion(**params).fit(R_cv,movieData)
#     mask = ~np.isnan(testData)
#     print(rmse(testData,model.predict(mask),mask))

In [ ]:
timer = time.time()
fusion = SimilarityFusion(**regu_params2).fit(R_cv,movieData)

print(rmse(fusion.predict(),testData,mask))
print(time.time()- timer)
finalParams = {"k_user":500, "k_movie":900, "delta":0.1, "alpha":0.3, "lam":0.4, "k_user_cross":100, "k_movie_cross":400, "reg_u":10, "reg_i":10,"n_iter":5 }
timer = time.time()
final = FusionKNN(**finalParams).fit(R_cv,movieData)
print(rmse(final.predict(),testData,mask))
print(time.time()- timer)




using global
  termes directs : 3.8s
  rectangle (prod) : 1.0s
0.8318336772358381
5.88791561126709
0.8318336772984624
4.946741104125977


In [15]:
output = np.load("output.npy")

mask = observed(valData)

rmse(output, valData,mask)

0.8740829984501383